# Breast Ultrasound Image Classification


In [2]:
import kagglehub
import shutil
import os

path = kagglehub.dataset_download("sabahesaraki/breast-ultrasound-images-dataset")

dest = os.path.join(os.getcwd(), "breast_ultrasound_dataset")

if not os.path.exists(dest):
    shutil.move(path, dest)
    print("Dataset saved in:", dest)
else:
    print("Dataset already exists at:", dest)

c:\Users\Asus\Desktop\Deep_learning_medical_images\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset already exists at: c:\Users\Asus\Desktop\Deep_learning_medical_images\tumor_classification(BUSI)\breast_ultrasound_dataset


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from collections import Counter
import numpy as np
from PIL import Image
import random
import matplotlib
matplotlib.use('Agg')  # non-interactive backend, safe for all environments
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

print("All imports successful.")

All imports successful.


In [4]:
SEED = 42

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

g = torch.Generator()
g.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [5]:
DATA_DIR = os.path.join(
    os.getcwd(),
    "breast_ultrasound_dataset",
    "Dataset_BUSI_with_GT"
)

def is_valid_file(path):
    return "_mask" not in path

dataset = ImageFolder(DATA_DIR, transform=None, is_valid_file=is_valid_file)

labels = [label for _, label in dataset.samples]
CLASS_NAMES = list(dataset.class_to_idx.keys())
print("Class mapping:", dataset.class_to_idx)
print("Dataset distribution:", Counter(labels))

Class mapping: {'benign': 0, 'malignant': 1, 'normal': 2}
Dataset distribution: Counter({0: 437, 1: 210, 2: 133})


In [6]:
train_idx, test_idx = train_test_split(
    range(len(labels)),
    test_size=0.2,
    stratify=labels,
    random_state=SEED
)

print(f"Train samples: {len(train_idx)}  |  Test samples: {len(test_idx)}")

Train samples: 624  |  Test samples: 156


In [7]:
augment_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x + 0.05 * torch.randn_like(x))
])

normal_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

print("Transforms defined.")

Transforms defined.


In [8]:
class AugmentedDataset(Dataset):

    def __init__(self, dataset, indices, labels):
        self.dataset = dataset
        self.indices = indices
        self.labels = labels
        self.extra_samples = []

        malignant = 1
        normal = 2

        malignant_idx = [i for i in indices if labels[i] == malignant]
        normal_idx    = [i for i in indices if labels[i] == normal]

        malignant_extra = int(len(malignant_idx) * 1.0)
        normal_extra    = int(len(normal_idx) * 1.0)

        random.seed(SEED)
        self.extra_samples += random.choices(malignant_idx, k=malignant_extra)
        self.extra_samples += random.choices(normal_idx,    k=normal_extra)

        print("Extra augmented samples:", len(self.extra_samples))

    def __len__(self):
        return len(self.indices) + len(self.extra_samples)

    def __getitem__(self, idx):
        if idx < len(self.indices):
            real_idx = self.indices[idx]
            path, label = self.dataset.samples[real_idx]
            img = Image.open(path).convert("RGB")
            return normal_transform(img), label
        else:
            aug_idx = self.extra_samples[idx - len(self.indices)]
            path, label = self.dataset.samples[aug_idx]
            img = Image.open(path).convert("RGB")
            return augment_transform(img), label


class TestDataset(Dataset):

    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        path, label = self.dataset.samples[real_idx]
        img = Image.open(path).convert("RGB")
        return normal_transform(img), label


print("Dataset classes defined.")

Dataset classes defined.


In [9]:
train_dataset = AugmentedDataset(dataset, train_idx, labels)
test_dataset  = TestDataset(dataset, test_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  generator=g)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, generator=g)

print(f"Train batches: {len(train_loader)}  |  Test batches: {len(test_loader)}")

Extra augmented samples: 274
Train batches: 29  |  Test batches: 5


In [10]:
class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = CNN().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

CNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (

In [25]:
class FocalLoss(nn.Module):

    def __init__(self, alpha=None, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss    = F.cross_entropy(inputs, targets, reduction='none')
        pt         = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.alpha is not None:
            focal_loss = self.alpha[targets] * focal_loss
        return focal_loss.mean()


alpha     = torch.tensor([1.0, 1.2, 2.0]).to(device)
criterion = FocalLoss(alpha=alpha, gamma=2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print("Loss function and optimizer ready.")

Loss function and optimizer ready.


In [26]:
EPOCHS = 15

history = {"epoch": [], "train_loss": [], "train_acc": []}

for epoch in range(EPOCHS):

    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels_batch in train_loader:
        images       = images.to(device)
        labels_batch = labels_batch.to(device)

        outputs = model(images)
        loss    = criterion(outputs, labels_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds       = torch.argmax(outputs, 1)
        correct    += (preds == labels_batch).sum().item()
        total      += labels_batch.size(0)

    epoch_loss = total_loss / len(train_loader)
    epoch_acc  = correct / total

    history["epoch"].append(epoch)
    history["train_loss"].append(epoch_loss)
    history["train_acc"].append(epoch_acc)

    print(f"Epoch {epoch:02d} | Loss {epoch_loss:.4f} | Train Acc {epoch_acc:.4f}")

print("\nTraining complete.")

Epoch 00 | Loss 0.3084 | Train Acc 0.7305
Epoch 01 | Loss 0.3028 | Train Acc 0.7261
Epoch 02 | Loss 0.2700 | Train Acc 0.7639
Epoch 03 | Loss 0.2627 | Train Acc 0.7461
Epoch 04 | Loss 0.2668 | Train Acc 0.7728
Epoch 05 | Loss 0.2892 | Train Acc 0.7394
Epoch 06 | Loss 0.2581 | Train Acc 0.7572
Epoch 07 | Loss 0.2615 | Train Acc 0.7617
Epoch 08 | Loss 0.2596 | Train Acc 0.7584
Epoch 09 | Loss 0.2601 | Train Acc 0.7795
Epoch 10 | Loss 0.2394 | Train Acc 0.7728
Epoch 11 | Loss 0.2511 | Train Acc 0.7739
Epoch 12 | Loss 0.2461 | Train Acc 0.7717
Epoch 13 | Loss 0.2697 | Train Acc 0.7695
Epoch 14 | Loss 0.2395 | Train Acc 0.7984

Training complete.


In [28]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ep = history["epoch"]

axes[0].plot(ep, history["train_loss"], color="#378ADD", linewidth=2,
             marker="o", markersize=4, label="Train loss")
axes[0].set_title("Training loss", fontsize=13)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Focal loss")
axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, history["train_acc"], color="#1D9E75", linewidth=2,
             marker="o", markersize=4, label="Train acc")
axes[1].set_title("Training accuracy", fontsize=13)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0.7, 0.85)
axes[1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle("Training curves", fontsize=15, y=1.02)
fig.tight_layout()

training_curve_path = "training_curves.png"
fig.savefig(training_curve_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", training_curve_path)

Saved: training_curves.png


C:\Users\Asus\AppData\Local\Temp\ipykernel_19628\1206123371.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [29]:
model.eval()

y_true, y_pred, y_prob = [], [], []

with torch.no_grad():
    for images, labels_batch in test_loader:
        images  = images.to(device)
        outputs = model(images)
        probs   = torch.softmax(outputs, dim=1)
        preds   = torch.argmax(outputs, dim=1)

        y_true.extend(labels_batch.numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

report = classification_report(y_true, y_pred, target_names=CLASS_NAMES)
conf   = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
roc    = roc_auc_score(y_true, y_prob, multi_class="ovr")

print(report)
print("Confusion Matrix:")
print(conf)
print(f"\nROC-AUC (macro OvR): {roc:.4f}")

              precision    recall  f1-score   support

      benign       0.76      0.77      0.77        87
   malignant       0.62      0.76      0.68        42
      normal       0.81      0.48      0.60        27

    accuracy                           0.72       156
   macro avg       0.73      0.67      0.68       156
weighted avg       0.73      0.72      0.71       156

Confusion Matrix:
[[67 17  3]
 [10 32  0]
 [11  3 13]]

ROC-AUC (macro OvR): 0.8677


In [30]:
y_true_bin = label_binarize(y_true, classes=[0, 1, 2])
colors = ["#378ADD", "#D85A30", "#1D9E75"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (cls_name, color) in enumerate(zip(CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc     = auc(fpr, tpr)

    axes[i].plot(fpr, tpr, color=color, linewidth=2, label=f"AUC = {roc_auc:.3f}")
    axes[i].plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5)
    axes[i].fill_between(fpr, tpr, alpha=0.08, color=color)
    axes[i].set_title(f"ROC — {cls_name}", fontsize=13)
    axes[i].set_xlabel("False positive rate")
    axes[i].set_ylabel("True positive rate")
    axes[i].set_xlim(0, 1)
    axes[i].set_ylim(0, 1.05)
    axes[i].legend(loc="lower right")
    axes[i].grid(True, alpha=0.3)

fig.suptitle(f"ROC curves  (macro OvR AUC = {roc:.4f})", fontsize=15, y=1.02)
fig.tight_layout()

roc_curve_path = "roc_curves.png"
fig.savefig(roc_curve_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", roc_curve_path)

Saved: roc_curves.png


C:\Users\Asus\AppData\Local\Temp\ipykernel_19628\1430892867.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [31]:
fig, ax = plt.subplots(figsize=(6, 5))

im = ax.imshow(conf, interpolation="nearest", cmap="Blues")
fig.colorbar(im, ax=ax)

tick_marks = np.arange(len(CLASS_NAMES))
ax.set_xticks(tick_marks)
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right")
ax.set_yticks(tick_marks)
ax.set_yticklabels(CLASS_NAMES)

thresh = conf.max() / 2.0
for i in range(conf.shape[0]):
    for j in range(conf.shape[1]):
        ax.text(j, i, str(conf[i, j]),
                ha="center", va="center", fontsize=12,
                color="white" if conf[i, j] > thresh else "black")

ax.set_title("Confusion matrix", fontsize=13)
ax.set_ylabel("True label")
ax.set_xlabel("Predicted label")
fig.tight_layout()

conf_matrix_path = "confusion_matrix.png"
fig.savefig(conf_matrix_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", conf_matrix_path)

Saved: confusion_matrix.png


C:\Users\Asus\AppData\Local\Temp\ipykernel_19628\1677011072.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [32]:
output_file = "focal_normal_results4.txt"

with open(output_file, "w") as f:
    f.write("Classification Report\n")
    f.write(report)
    f.write("\n\nConfusion Matrix\n")
    f.write(str(conf))
    f.write("\n\nROC-AUC\n")
    f.write(str(roc))
    f.write("\n\nSaved plots\n")
    f.write(f"  Training curves : {training_curve_path}\n")
    f.write(f"  ROC curves      : {roc_curve_path}\n")
    f.write(f"  Confusion matrix: {conf_matrix_path}\n")

print("Results saved to:", output_file)
print("\nAll output files:")
for f in [output_file, training_curve_path, roc_curve_path, conf_matrix_path]:
    print(" ", f)

Results saved to: focal_normal_results4.txt

All output files:
  focal_normal_results4.txt
  training_curves.png
  roc_curves.png
  confusion_matrix.png
